
Complete Drift Detection Pipeline

Production-ready drift monitoring with PSI and KS tests



In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

# Sample X_train data (reference data)
X_train = pd.DataFrame({
    'feature_a': np.random.normal(loc=0, scale=1, size=1000),
    'feature_b': np.random.normal(loc=5, scale=2, size=1000),
    'feature_c': np.random.randint(0, 10, size=1000)
})

# Sample X_production data (current data with some drift)
X_production = pd.DataFrame({
    'feature_a': np.random.normal(loc=0.5, scale=1.2, size=1000), # Drifted
    'feature_b': np.random.normal(loc=5, scale=2, size=1000), # Stable
    'feature_c': np.random.randint(2, 12, size=1000) # Drifted
})

In [2]:
import numpy as np
import pandas as pd
from scipy import stats
from datetime import datetime

# ============================================================
# DRIFT DETECTION: PSI + KS Test
# Use daily on production features to catch distribution shifts
# ============================================================

class DriftDetector:
  """Detect data drift using PSI and KS tests."""

  def __init__(self, reference_data, psi_threshold=0.1, ks_alpha=0.05):
    self.reference = reference_data
    self.psi_threshold = psi_threshold
    self.ks_alpha = ks_alpha
    self.history = []

  def calculate_psi(self, expected, actual, bins=10):
    """Population Stability Index for drift detection.
        PSI < 0.1: stable | 0.1-0.25: monitor | > 0.25: action
    """
    breakpoints = np.percentile(expected, np.linspace(0, 100, bins + 1))
    breakpoints[0], breakpoints[-1] = -np.inf, np.inf

    exp_pct = np.histogram(expected, breakpoints)[0] / len(expected)
    act_pct = np.histogram(actual, breakpoints)[0] / len(actual)

    # Avoid log(0): clip to small value
    exp_pct = np.clip(exp_pct, 0.0001, None)
    act_pct = np.clip(act_pct, 0.0001, None)

    return np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct))

  def check_drift(self, current_data):
    """Check all features for drift, return structured report"""
    report = {
        'timestamp': datetime.now().isoformat(),
        'features' : {},
        'alert_level' : 'OK' # OK, WARNING, CRITICAL
    }

    max_severity = 0
    for col in self.reference.columns:
      if col not in current_data.columns:
        continue

      ref_col = self.reference[col].dropna()
      cur_col = current_data[col].dropna()

      psi = self.calculate_psi(ref_col, cur_col)
      ks_stat, p_value = stats.ks_2samp(ref_col, cur_col)

      # detemine severity

      if psi > 0.25:
        severity = 'CRITICAL'
        max_severity = max(max_severity,2)
      elif psi > self.psi_threshold:
        severity = 'WARNING'
        max_severity = max(max_severity,1)
      else:
        severity = 'OK'

      report['features'][col] = {
          'psi' : round(psi, 4),
          'ks_statistic': round(ks_stat,4),
          'p_value' : round(p_value,4),
          'severity' : severity
      }

      report['alert_level'] = ['OK', 'WARNING', 'CRITICAL'][max_severity]
      self.history.append(report)
      return report

# Usage

detector = DriftDetector(X_train, psi_threshold=0.1)
report = detector.check_drift(X_production)
if report['alert_level'] != 'OK':
    for feat, m in report['features'].items():
        if m['severity'] != 'OK':
            print(f"[{m['severity']}] {feat}: PSI={m['psi']}")


[WARNING] feature_a: PSI=0.2348


Model Performance Monitor with Alerts

Track predictions, latency, accuracy, and trigger severity-based alerts




In [5]:
import numpy as np
from collections import deque
from datetime import datetime
import time

# ============================================================
# MODEL MONITOR: Track predictions and trigger alerts
# Implements the three-pillar monitoring approach
# ============================================================

class ModelMonitor:
  """ Monitor Model performance in production..."""

  def __init__(self, window_size=1000):
    self.predictions = deque(maxlen=window_size)
    self.latencies = deque(maxlen=window_size)
    self.actuals = deque(maxlen=window_size)
    self.alerts = []

    # Thresholds (tune based on your model)
    self.thresholds = {
        'latency_p99_ms': 100,
        'accuracy_min': 0.85
    }

  def log_prediction(self, prediction, latency_ms, actual=None):
    """Log a single prediction with optional ground truth."""
    self.predictions.append(prediction)
    self.latencies.append(latency_ms)
    if actual is not None:
      self.actuals.append((prediction, actual))

  def get_metrics(self):
    """ Calculate current monitoring metrics"""
    metrics = {
        'timestamp': datetime.now().isoformat(),
        'prediction_count' : len(self.predictions)
    }

    # latency metrics
    if self.latencies:
      lats = list(self.latencies)
      metrics['latency_p50'] = round(np.percentile(lats, 50), 1)
      metrics['latency_p99'] = round(np.percentile(lats, 99), 1)

    # accuracy (when labels available)
    if len(self.actuals) > 100:
      correct = sum(1 for p, a in self.actuals if p==a)
      metrics['accuracy'] = round (correct / len(self.actuals), 4)

    return metrics

  def check_alerts(self):
    """Check thresholds and return alerts with severity"""
    metrics = self.get_metrics()
    alerts = []

    # latency check
    p99 = metrics.get('latency_p99', 0)
    threshold = self.thresholds['latency_p99_ms']
    if p99 > threshold * 2:
      alerts.append({
          'severity': 'CRITICAL',
          'type' : 'LATENCY',
          'message' : f"p99={p99:.0f}ms (>{threshold*2}ms)"
      })

    elif p99 > threshold:
      alerts.append({
          'severity': 'WARNING',
          'type' : 'LATENCY',
          'message': f"p99={p99:.0f}ms (>{threshold}ms)"
      })

    # Accuracy check
    acc = metrics.get('accuracy')
    if acc is not None and acc < self.thresholds['accuracy_min']:
      alerts.append({
          'severity': 'CRITICAL',
          'type' : 'ACCURACY',
          'message' : f"Accuracy={acc:.1%}"
      })

    self.alerts.extend(alerts)
    return alerts


# usage
monitor = ModelMonitor(window_size=5000)

# Dummy data for demonstration
class DummyModel:
    def predict(self, features):
        return np.random.randint(0, 2)

model = DummyModel()
production_stream = range(1000) # Simulate 1000 requests

y_production_actual = np.random.randint(0, 2, size=1000) # Dummy actuals

for i, request in enumerate(production_stream):
  start = time.time()
  pred = model.predict(X_production.iloc[[i]]) # Assuming X_production is defined
  latency = (time.time() - start) * 1000
  monitor.log_prediction(pred, latency, actual=y_production_actual[i])

alerts = monitor.check_alerts()
for a in alerts:
  print(f"[{a['severity']}]  [{a['type']}] : {a['message']}")

metrics = monitor.get_metrics()
print("\n--- Current Metrics ---")
for key, value in metrics.items():
    print(f"{key}: {value}")


[CRITICAL]  [ACCURACY] : Accuracy=48.6%

--- Current Metrics ---
timestamp: 2026-08-25T11:18:32.817461
prediction_count: 1000
latency_p50: 0.2
latency_p99: 0.3
accuracy: 0.486


Safe Retraining with Champion vs Challenger

Validate retrained models before deployment using comparison testing

In [7]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from datetime import datetime

# ============================================================
# SAFE RETRAINING: Champion vs Challenger
# Never deploy without comparing to production model
# ============================================================

class SafeRetrainer:
  """ Validate retrained models before deployment"""

  def __init__(self, champion_model, min_improvement=0.02):
    self.champion = champion_model
    self.min_improvement = min_improvement
    self.history = []

  def evaluate_model(self, model, X_test, y_test):
    """Compute standard evaluation metrics"""
    preds = model.predict(X_test)
    return {
        'accuracy' : accuracy_score(y_test, preds),
        'f1' : f1_score(y_test, preds, average = 'weighted')
    }

  def compare_models(self, challenger, X_test, y_test):
    champ_metrics = self.evaluate_model(self.champion, X_test, y_test)
    chall_metrics = self.evaluate_model(challenger, X_test, y_test)
    improvement = chall_metrics['accuracy'] - champ_metrics['accuracy']

    result = {
        'timestamp' : datetime.now().isoformat(),
        'champion' : champ_metrics,
        'challenger' : chall_metrics,
        'improvement' : round(improvement,4),
        'promote' : improvement >= self.min_improvement,
    }

    if result['promote']:
      result['reason'] = f"Challenger wins: +{improvement:.1%} accuracy"
    else:
      result['reason'] = (
          f"Champion Retained: {improvement:.1%} (below min {self.min_improvement:.1%})"
      )
    self.history.append(result)
    return result

  def promote_challenger(self, challenger):
    """Promote challenger toc champion (after validation)"""
    self.champion = challenger
    return "Challenger updated. Monitor closly for 24 - 48h"

# Dummy data for demonstration
class DummyModel:
    def __init__(self, accuracy_bias=0):
        self.accuracy_bias = accuracy_bias

    def predict(self, features):
        # Simulate predictions with a slight bias for the challenger
        num_samples = features.shape[0]
        if np.random.rand() < (0.5 + self.accuracy_bias):
            return np.random.randint(0, 2, size=num_samples)
        else:
            return np.random.randint(0, 2, size=num_samples)

# Instantiate dummy models
production_model = DummyModel(accuracy_bias=0) # Baseline accuracy
new_model = DummyModel(accuracy_bias=0.05) # Slightly better accuracy

X_holdout = X_production # Use existing production data as holdout
y_holdout = y_production_actual # Use existing actual labels as holdout

# usage
retrainer = SafeRetrainer(production_model, min_improvement=0.02)

# after retraining on freshdata
result = retrainer.compare_models(new_model, X_holdout, y_holdout)
print(f"Decision: {result['reason']}")

if(result['promote']):
  print(retrainer.promote_challenger(new_model))
  # Begin gradual rollout: 10% -> 50% -> 100%

Decision: Champion Retained: 1.7% (below min 2.0%)
